In [ ]:
from __future__ import annotations

import ast
import json
import sqlite3
from pathlib import Path

import pandas as pd

pd.options.mode.copy_on_write = True

# Optional faster JSON parser: use orjson if installed, otherwise fallback to stdlib
try:
    import orjson as _orjson
    _json_loads = _orjson.loads
except Exception:
    _json_loads = json.loads

DB_PATH = Path('moltbook_27_30.db')
OUTPUT_DIR = Path('.')
START_DATE = '2026-01-27'
CUTOFF_DATE = '2026-03-21'


In [ ]:
def connect_sqlite_read_only(db_path: Path) -> sqlite3.Connection:
    """Opens the SQLite database in strictly read-only and query-only mode."""
    if not db_path.exists():
        raise FileNotFoundError(f'Database not found: {db_path}')

    uri = f'file:{db_path.as_posix()}?mode=ro&immutable=1'
    conn = sqlite3.connect(uri, uri=True)
    conn.execute('PRAGMA query_only = ON')
    conn.execute('PRAGMA foreign_keys = ON')
    return conn


def not_deleted_sql(alias: str) -> str:
    """
    SQL predicate to exclude deleted rows, compatible with common boolean representations in SQLite (1/0, true/false, NULL).
    """
    return (
        f'({alias}.is_deleted IS NULL '
        f"OR (LOWER(CAST({alias}.is_deleted AS TEXT)) NOT IN ('1', 'true')))"
    )


def print_basic_stats(name: str, df: pd.DataFrame, user_col: str) -> None:
    """Prints basic statistics about the dataset."""
    if df.empty:
        print(f'[{name}] Empty dataset')
        return

    min_day = df['day'].min().date()
    max_day = df['day'].max().date()
    unique_users = df[user_col].nunique(dropna=True)

    print(f'[{name}] rows: {len(df):,}')
    print(f'[{name}] temporal range: {min_day} -> {max_day}')
    print(f'[{name}] unique users ({user_col}): {unique_users:,}')

In [ ]:
def _normalize_optional_string(value: object) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def _is_deleted_value(value: object) -> bool:
    if value is None:
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)) and not pd.isna(value):
        return int(value) == 1
    return str(value).strip().lower() in {'1', 'true', 't', 'yes'}


def _safe_literal_eval_replies(raw_replies: object) -> list[dict]:
    if raw_replies is None:
        return []
    if isinstance(raw_replies, float) and pd.isna(raw_replies):
        return []
    if isinstance(raw_replies, str):
        payload = raw_replies.strip()
        if payload in {'', '[]', 'None', 'null'}:
            return []
        try:
            parsed = ast.literal_eval(payload)
        except (SyntaxError, ValueError):
            return []
    else:
        parsed = raw_replies
    if parsed is None:
        return []
    if isinstance(parsed, dict):
        return [parsed]
    if isinstance(parsed, list):
        return [item for item in parsed if isinstance(item, dict)]
    return []


def _parse_raw_json_metrics(raw: object) -> tuple[float | None, float | None]:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None, None
    if isinstance(raw, (dict, list)):
        data = raw
    else:
        text = str(raw).strip()
        if not text:
            return None, None
        try:
            data = _json_loads(text)
        except Exception:
            try:
                data = ast.literal_eval(text)
            except Exception:
                return None, None
    if not isinstance(data, dict):
        return None, None
    author = data.get('author') if isinstance(data.get('author'), dict) else None
    karma = None
    followers = None
    if author is not None:
        karma = author.get('karma') if author.get('karma') is not None else author.get('author_karma')
        followers = author.get('followerCount') if author.get('followerCount') is not None else author.get('follower_count') if author.get('follower_count') is not None else author.get('followers')
    if karma is None:
        karma = data.get('karma') if data.get('karma') is not None else data.get('author_karma')
    if followers is None:
        followers = data.get('followerCount') if data.get('followerCount') is not None else data.get('follower_count')
    try:
        karma_value = float(karma) if karma is not None else None
    except Exception:
        karma_value = None
    try:
        follower_value = float(followers) if followers is not None else None
    except Exception:
        follower_value = None
    return karma_value, follower_value


def _extract_reply_author(reply_obj: dict) -> tuple[str | None, str | None, str | None, float | None, float | None]:
    author_obj = reply_obj.get('author') if isinstance(reply_obj.get('author'), dict) else {}
    author_id = _normalize_optional_string(reply_obj.get('author_id') or author_obj.get('id'))
    author_username = _normalize_optional_string(author_obj.get('username') or author_obj.get('user_name') or author_obj.get('handle'))
    author_display_name = _normalize_optional_string(author_obj.get('display_name') or author_obj.get('name') or author_obj.get('full_name') or author_obj.get('nickname'))
    karma_candidate = author_obj.get('karma') if author_obj.get('karma') is not None else author_obj.get('author_karma')
    followers_candidate = author_obj.get('followerCount') if author_obj.get('followerCount') is not None else author_obj.get('follower_count') if author_obj.get('follower_count') is not None else author_obj.get('followers')
    try:
        author_karma = float(karma_candidate) if karma_candidate is not None else None
    except Exception:
        author_karma = None
    try:
        author_followers = float(followers_candidate) if followers_candidate is not None else None
    except Exception:
        author_followers = None
    if author_karma is None and reply_obj.get('karma') is not None:
        try:
            author_karma = float(reply_obj.get('karma'))
        except Exception:
            pass
    if author_followers is None and reply_obj.get('followerCount') is not None:
        try:
            author_followers = float(reply_obj.get('followerCount'))
        except Exception:
            pass
    return author_id, author_username, author_display_name, author_karma, author_followers


def _collect_replies_recursive(
    replies_payload: object,
    *,
    post_id: object,
    root_comment_id: object,
    parent_id: object,
    post_author_id: object,
    out_rows: list[dict],
    fallback_depth: int = 1,
 ) -> None:
    for reply in _safe_literal_eval_replies(replies_payload):
        reply_id = _normalize_optional_string(reply.get('id'))
        reply_parent_id = _normalize_optional_string(reply.get('parent_id') or parent_id)
        reply_root_comment_id = _normalize_optional_string(reply.get('root_comment_id') or root_comment_id)
        reply_post_id = _normalize_optional_string(reply.get('post_id') or post_id)
        author_id, author_username, author_display_name, author_karma, author_followers = _extract_reply_author(reply)
        depth_value = reply.get('depth')
        try:
            normalized_depth = int(depth_value) if depth_value is not None else int(fallback_depth)
        except (TypeError, ValueError):
            normalized_depth = int(fallback_depth)
        row = {
            'id': reply_id,
            'post_id': reply_post_id,
            'parent_id': reply_parent_id,
            'root_comment_id': reply_root_comment_id,
            'content': reply.get('content'),
            'author_id': author_id,
            'created_at': reply.get('created_at'),
            'updated_at': reply.get('updated_at'),
            'depth': normalized_depth,
            'score': reply.get('score'),
            'upvotes': reply.get('upvotes'),
            'downvotes': reply.get('downvotes'),
            'verification_status': reply.get('verification_status'),
            'is_spam': reply.get('is_spam'),
            'is_deleted': reply.get('is_deleted'),
            'post_author_id': _normalize_optional_string(post_author_id),
            'author_username': author_username,
            'author_display_name': author_display_name,
            'author_karma': author_karma,
            'author_followers': author_followers,
        }
        out_rows.append(row)
        child_parent_id = row['id'] if row['id'] is not None else row['parent_id']
        _collect_replies_recursive(
            reply.get('replies'),
            post_id=row['post_id'],
            root_comment_id=row['root_comment_id'],
            parent_id=child_parent_id,
            post_author_id=row['post_author_id'],
            out_rows=out_rows,
            fallback_depth=normalized_depth + 1,
        )


def fetch_recursive_replies(conn: sqlite3.Connection, start_date: str = START_DATE, cutoff_date: str = CUTOFF_DATE) -> pd.DataFrame:
    query = f'''
        SELECT c.id AS root_comment_id,
               c.post_id AS post_id,
               c.replies AS replies,
               CAST(p.author_id AS TEXT) AS post_author_id
        FROM comments_new AS c
        INNER JOIN posts AS p ON c.post_id = p.id
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
          AND c.replies IS NOT NULL
    '''
    base_df = pd.read_sql_query(query, conn, params=(start_date, cutoff_date))
    records: list[dict] = []
    for row in base_df.itertuples(index=False):
        _collect_replies_recursive(
            row.replies,
            post_id=row.post_id,
            root_comment_id=row.root_comment_id,
            parent_id=row.root_comment_id,
            post_author_id=row.post_author_id,
            out_rows=records,
            fallback_depth=1,
        )
    columns = ['id', 'post_id', 'parent_id', 'root_comment_id', 'content', 'author_id', 'created_at', 'updated_at', 'depth', 'score', 'upvotes', 'downvotes', 'verification_status', 'is_spam', 'is_deleted', 'post_author_id', 'author_username', 'author_display_name', 'author_karma', 'author_followers', 'day', 'source', 'target']
    if not records:
        return pd.DataFrame(columns=columns)
    reply_df = pd.DataFrame(records)
    reply_df['author_id'] = reply_df['author_id'].astype('string').str.strip()
    reply_df['post_author_id'] = reply_df['post_author_id'].astype('string').str.strip()
    # Parse created_at with UTC awareness to avoid mixed timezone errors
    reply_df['created_at'] = pd.to_datetime(reply_df['created_at'], errors='coerce', utc=True)
    reply_df['day'] = reply_df['created_at'].dt.tz_convert(None).dt.normalize()
    reply_df['source'] = reply_df['author_id']
    reply_df['target'] = reply_df['post_author_id']
    reply_df = reply_df[reply_df['day'].notna()]
    reply_df = reply_df[~reply_df['is_deleted'].map(_is_deleted_value)]
    reply_df = reply_df.dropna(subset=['source', 'target'])
    reply_df = reply_df[reply_df['source'] != reply_df['target']]
    reply_df = reply_df.sort_values(['day', 'source', 'target', 'id']).reset_index(drop=True)
    for col in columns:
        if col not in reply_df.columns:
            reply_df[col] = pd.NA
    return reply_df[columns]


def fetch_interactions(conn: sqlite3.Connection, start_date: str = START_DATE, cutoff_date: str = CUTOFF_DATE, replies_df: pd.DataFrame | None = None) -> pd.DataFrame:
    query = f'''
        SELECT DATE(c.created_at) AS day,
               CAST(c.author_id AS TEXT) AS source,
               CAST(p.author_id AS TEXT) AS target,
               CAST(c.id AS TEXT) AS interaction_id,
               CAST(c.post_id AS TEXT) AS post_id,
               'comment' AS interaction_type
        FROM comments_new AS c
        INNER JOIN posts AS p ON c.post_id = p.id
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
          AND c.author_id IS NOT NULL
          AND p.author_id IS NOT NULL
    '''
    df = pd.read_sql_query(query, conn, params=(start_date, cutoff_date))
    if replies_df is not None and not replies_df.empty:
        reply_interactions = replies_df[['day', 'source', 'target', 'id', 'post_id']].copy()
        reply_interactions = reply_interactions.rename(columns={'id': 'interaction_id'})
        reply_interactions['interaction_type'] = 'reply'
        df = pd.concat([df, reply_interactions], ignore_index=True, sort=False)
    if df.empty:
        return pd.DataFrame(columns=['day', 'source', 'target', 'interaction_id', 'post_id', 'interaction_type'])
    # Handle mixed timezones by parsing to UTC then converting to naive UTC midnight
    df['day'] = pd.to_datetime(df['day'], errors='coerce', utc=True)
    df['day'] = df['day'].dt.tz_convert(None).dt.normalize()
    df['source'] = df['source'].astype('string').str.strip()
    df['target'] = df['target'].astype('string').str.strip()
    df = df.dropna(subset=['day', 'source', 'target'])
    df = df[df['source'] != df['target']]
    df = df.sort_values(['day', 'source', 'target', 'interaction_id']).reset_index(drop=True)
    return df


In [ ]:


def fetch_active_users(conn: sqlite3.Connection, start_date: str = START_DATE, cutoff_date: str = CUTOFF_DATE, replies_df: pd.DataFrame | None = None) -> pd.DataFrame:
    posts_q = f'''
        SELECT DATE(p.created_at) AS day, CAST(p.author_id AS TEXT) AS user_id, 'post' AS source_type
        FROM posts AS p
        WHERE DATE(p.created_at) >= ?
          AND DATE(p.created_at) <= ?
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
    '''
    comments_q = f'''
        SELECT DATE(c.created_at) AS day, CAST(c.author_id AS TEXT) AS user_id, 'comment' AS source_type
        FROM comments_new AS c
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND c.author_id IS NOT NULL
    '''
    posts_df = pd.read_sql_query(posts_q, conn, params=(start_date, cutoff_date))
    comments_df = pd.read_sql_query(comments_q, conn, params=(start_date, cutoff_date))
    frames = [posts_df, comments_df]
    if replies_df is not None and not replies_df.empty:
        reply_users = replies_df[['day', 'author_id']].copy()
        reply_users = reply_users.rename(columns={'author_id': 'user_id'})
        reply_users['source_type'] = 'reply'
        frames.append(reply_users)
    df = pd.concat(frames, ignore_index=True, sort=False)
    if df.empty:
        return pd.DataFrame(columns=['day', 'user_id', 'source_type'])
    # parse day with UTC awareness to avoid mixed timezone issues
    df['day'] = pd.to_datetime(df['day'], errors='coerce', utc=True)
    df['day'] = df['day'].dt.tz_convert(None).dt.normalize()
    df['user_id'] = df['user_id'].astype('string').str.strip()
    df = df.dropna(subset=['day', 'user_id'])
    df = df.sort_values(['day', 'user_id', 'source_type']).drop_duplicates(subset=['day', 'user_id'], keep='first').reset_index(drop=True)
    return df

In [ ]:
def fetch_daily_author_metrics(conn: sqlite3.Connection, start_date: str = START_DATE, cutoff_date: str = CUTOFF_DATE) -> pd.DataFrame:
    """Extracts each record from (day, author_id, karma, followerCount) posts and comments_new.
    Returns a table with columns ['day','author_id','karma','followerCount'].
    """
    query = f"""
        SELECT DATE(p.created_at) AS day,
               CAST(p.author_id AS TEXT) AS author_id,
               CAST(json_extract(p.raw_json, '$.author.karma') AS REAL) AS karma,
               CAST(json_extract(p.raw_json, '$.author.followerCount') AS REAL) AS followerCount
        FROM posts AS p
        WHERE DATE(p.created_at) >= ?
          AND DATE(p.created_at) <= ?
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
        
        UNION ALL
        
        SELECT DATE(c.created_at) AS day,
               CAST(c.author_id AS TEXT) AS author_id,
               CAST(json_extract(c.raw_json, '$.author.karma') AS REAL) AS karma,
               CAST(json_extract(c.raw_json, '$.author.followerCount') AS REAL) AS followerCount
        FROM comments_new AS c
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND c.author_id IS NOT NULL
    """
    df = pd.read_sql_query(query, conn, params=(start_date, cutoff_date, start_date, cutoff_date))
    if df.empty:
        return pd.DataFrame(columns=['day','author_id','karma','followerCount'])
    # parse day with UTC awareness to avoid mixed timezone issues
    df['day'] = pd.to_datetime(df['day'], errors='coerce', utc=True)
    df['day'] = df['day'].dt.tz_convert(None).dt.normalize()
    df['author_id'] = df['author_id'].astype('string').str.strip()
    
    df['karma'] = pd.to_numeric(df['karma'], errors='coerce')
    df['followerCount'] = pd.to_numeric(df['followerCount'], errors='coerce')
    df = df.dropna(subset=['day','author_id'])
    return df[['day','author_id','karma','followerCount']]

In [ ]:
def run_pipeline(
    db_path: Path,
    output_dir: Path,
    start_date: str = START_DATE,
    cutoff_date: str = CUTOFF_DATE,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
    with connect_sqlite_read_only(db_path) as conn:
        replies_df = fetch_recursive_replies(conn, start_date=start_date, cutoff_date=cutoff_date)
        interactions = fetch_interactions(
            conn,
            start_date=start_date,
            cutoff_date=cutoff_date,
            replies_df=replies_df,
        )
        users = fetch_active_users(
            conn,
            start_date=start_date,
            cutoff_date=cutoff_date,
            replies_df=replies_df,
        )

    start_day = pd.Timestamp(start_date).normalize()
    cutoff_day = pd.Timestamp(cutoff_date).normalize()

    if not interactions.empty:
        interactions = interactions.copy()
        interactions['day'] = pd.to_datetime(interactions['day'], errors='coerce', utc=True)
        interactions['day'] = interactions['day'].dt.tz_convert(None).dt.normalize()
        interactions = interactions[(interactions['day'] >= start_day) & (interactions['day'] <= cutoff_day)].reset_index(drop=True)

    if not users.empty:
        users = users.copy()
        users['day'] = pd.to_datetime(users['day'], errors='coerce', utc=True)
        users['day'] = users['day'].dt.tz_convert(None).dt.normalize()
        users = users[(users['day'] >= start_day) & (users['day'] <= cutoff_day)].reset_index(drop=True)

    interactions_path = output_dir / 'interactions.parquet'
    users_path = output_dir / 'users.parquet'

    # Optimization: snappy compression + pyarrow engine for faster IO on large datasets
    interactions.to_parquet(interactions_path, index=False, compression='snappy', engine='pyarrow')
    users.to_parquet(users_path, index=False, compression='snappy', engine='pyarrow')

    print(f'Salvato: {interactions_path.resolve()}')
    print(f'Salvato: {users_path.resolve()}')

    print_basic_stats('interactions', interactions, user_col='source')
    print_basic_stats('users', users, user_col='user_id')

    return interactions, users

In [ ]:
interactions_df, users_df = run_pipeline(DB_PATH, OUTPUT_DIR, START_DATE, CUTOFF_DATE)

display(interactions_df.head())
display(users_df.head())

In [ ]:
def compute_preferential_attachment_timeseries(
    output_dir: Path = OUTPUT_DIR,
    top_percents: tuple[int, ...] = tuple(range(1, 21)),
    start_date: str = START_DATE,
    cutoff_date: str = CUTOFF_DATE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Computes the preferential attachment time series using the already generated interactions/users parquet files."""
    import numpy as np

    interactions_path = output_dir / 'interactions.parquet'
    users_path = output_dir / 'users.parquet'

    interactions = pd.read_parquet(interactions_path)
    users = pd.read_parquet(users_path)

    if interactions.empty or users.empty:
        empty_timeseries = pd.DataFrame(columns=['date', 'top_percent', 'n_new_edges', 'n_new_edges_to_top', 'p_new_edges_to_top'])
        empty_summary = pd.DataFrame(columns=['top_percent', 'mean_p_new_edges_to_top', 'std_p_new_edges_to_top', 'n_days'])
        return empty_timeseries, empty_summary

    interactions = interactions.copy()
    users = users.copy()

    # Parse day columns with UTC awareness to avoid mixed timezone errors
    interactions['day'] = pd.to_datetime(interactions['day'], errors='coerce', utc=True)
    interactions['day'] = interactions['day'].dt.tz_convert(None).dt.normalize()
    users['day'] = pd.to_datetime(users['day'], errors='coerce', utc=True)
    users['day'] = users['day'].dt.tz_convert(None).dt.normalize()

    start_day = pd.Timestamp(start_date).normalize()
    cutoff_day = pd.Timestamp(cutoff_date).normalize()
    interactions = interactions[(interactions['day'] >= start_day) & (interactions['day'] <= cutoff_day)].reset_index(drop=True)
    users = users[(users['day'] >= start_day) & (users['day'] <= cutoff_day)].reset_index(drop=True)

    if interactions.empty or users.empty:
        empty_timeseries = pd.DataFrame(columns=['date', 'top_percent', 'n_new_edges', 'n_new_edges_to_top', 'p_new_edges_to_top'])
        empty_summary = pd.DataFrame(columns=['top_percent', 'mean_p_new_edges_to_top', 'std_p_new_edges_to_top', 'n_days'])
        return empty_timeseries, empty_summary

    interactions = interactions.dropna(subset=['day', 'source', 'target'])
    users = users.dropna(subset=['day', 'user_id'])

    interactions['source'] = interactions['source'].astype('string').str.strip()
    interactions['target'] = interactions['target'].astype('string').str.strip()
    users['user_id'] = users['user_id'].astype('string').str.strip()

    
    edge_first_seen = (
        interactions.groupby(['source', 'target'], as_index=False)['day']
        .min()
        .rename(columns={'day': 'first_seen_day'})
        .sort_values(['first_seen_day', 'source', 'target'])
        .reset_index(drop=True)
    )

    # Existing agents: first day on which they appear in the users parquet file or as endpoints of interactions.
    user_first_seen = users.groupby('user_id', as_index=True)['day'].min()
    endpoints = pd.concat(
        [
            interactions[['day', 'source']].rename(columns={'source': 'user_id'}),
            interactions[['day', 'target']].rename(columns={'target': 'user_id'}),
        ],
        ignore_index=True,
    )
    endpoint_first_seen = endpoints.groupby('user_id', as_index=True)['day'].min()
    agent_first_seen = pd.concat([user_first_seen, endpoint_first_seen], axis=1).min(axis=1)
    agent_first_seen = agent_first_seen.dropna().sort_values()

    days = pd.date_range(
        min(edge_first_seen['first_seen_day'].min(), agent_first_seen.min()),
        max(edge_first_seen['first_seen_day'].max(), agent_first_seen.max()),
        freq='D',
    )

    # I group the edges by day to avoid repeated filters
    edges_by_day = {day: frame.copy() for day, frame in edge_first_seen.groupby('first_seen_day') }
    agents_by_day: dict[pd.Timestamp, list[str]] = {}
    for user_id, first_day in agent_first_seen.items():
        agents_by_day.setdefault(pd.Timestamp(first_day).normalize(), []).append(str(user_id))

    active_nodes: set[str] = set()
    cumulative_indegree: dict[str, int] = {}
    pref_rows: list[dict] = []

    for day in days:
        day = pd.Timestamp(day).normalize()
        current_edges = edges_by_day.get(day)
        n_new_edges = int(len(current_edges)) if current_edges is not None else 0

        if active_nodes and n_new_edges > 0:
            indegree_series = pd.Series({node: cumulative_indegree.get(node, 0) for node in active_nodes}, dtype='int64')
            for top_percent in top_percents:
                top_n = max(1, int(np.ceil(len(indegree_series) * top_percent / 100.0)))
                top_nodes = set(indegree_series.sort_values(ascending=False, kind='mergesort').head(top_n).index)
                n_new_edges_to_top = int(current_edges['target'].isin(top_nodes).sum())
                p_value = n_new_edges_to_top / n_new_edges if n_new_edges > 0 else np.nan
                pref_rows.append({
                    'date': day,
                    'top_percent': top_percent,
                    'n_new_edges': n_new_edges,
                    'n_new_edges_to_top': n_new_edges_to_top,
                    'p_new_edges_to_top': p_value,
                })
        else:
            for top_percent in top_percents:
                pref_rows.append({
                    'date': day,
                    'top_percent': top_percent,
                    'n_new_edges': n_new_edges,
                    'n_new_edges_to_top': 0,
                    'p_new_edges_to_top': np.nan,
                })

        # Updates the cumulative graph for the following day without leakage.
        if day in agents_by_day:
            active_nodes.update(agents_by_day[day])
        if current_edges is not None and not current_edges.empty:
            for target in current_edges['target'].astype('string'):
                cumulative_indegree[target] = cumulative_indegree.get(target, 0) + 1
            active_nodes.update(current_edges['source'].astype('string'))
            active_nodes.update(current_edges['target'].astype('string'))

    pref_attach_timeseries = pd.DataFrame(pref_rows)
    # normalize date column with UTC awareness
    pref_attach_timeseries['date'] = pd.to_datetime(pref_attach_timeseries['date'], utc=True)
    pref_attach_timeseries['date'] = pref_attach_timeseries['date'].dt.tz_convert(None).dt.normalize()
    pref_attach_timeseries = pref_attach_timeseries.sort_values(['date', 'top_percent']).reset_index(drop=True)

    valid = pref_attach_timeseries[(pref_attach_timeseries['n_new_edges'] > 0) & pref_attach_timeseries['p_new_edges_to_top'].notna()]
    pref_attach_summary = (
        valid.groupby('top_percent', as_index=False)
        .agg(
            mean_p_new_edges_to_top=('p_new_edges_to_top', 'mean'),
            std_p_new_edges_to_top=('p_new_edges_to_top', 'std'),
            n_days=('p_new_edges_to_top', 'count'),
        )
        .sort_values('top_percent')
        .reset_index(drop=True)
    )

    return pref_attach_timeseries, pref_attach_summary

In [ ]:
day_means_cum_full = pd.DataFrame(columns=['karma', 'followerCount'])

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

from scipy.optimize import minimize_scalar
from scipy.special import zeta


ROBUSTNESS_RANDOM_RUNS = 3
REMOVAL_PERCENTS = (0.01, 0.05, 0.10, 0.20)
PERCENTILES = (5, 25, 50, 75, 95)
EXCEL_OUTPUT = OUTPUT_DIR / "graph_dynamics_timeseries.xlsx"


def fetch_daily_activity_counts(
    conn: sqlite3.Connection,
    start_date: str = START_DATE,
    cutoff_date: str = CUTOFF_DATE,
    replies_df: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Counts posts/comments per (day, user), using SQL aggregation to reduce memory usage.
    If replies_df is provided, replies are counted as comments."""
    posts_q = f"""
        SELECT DATE(p.created_at) AS day,
               CAST(p.author_id AS TEXT) AS user_id,
               COUNT(*) AS num_posts
        FROM posts AS p
        WHERE DATE(p.created_at) >= ?
          AND DATE(p.created_at) <= ?
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
        GROUP BY DATE(p.created_at), CAST(p.author_id AS TEXT)
    """

    comments_q = f"""
        SELECT DATE(c.created_at) AS day,
               CAST(c.author_id AS TEXT) AS user_id,
               COUNT(*) AS num_comments
        FROM comments_new AS c
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND c.author_id IS NOT NULL
        GROUP BY DATE(c.created_at), CAST(c.author_id AS TEXT)
    """

    posts_df = pd.read_sql_query(posts_q, conn, params=(start_date, cutoff_date))
    comments_df = pd.read_sql_query(comments_q, conn, params=(start_date, cutoff_date))

    for df, col in ((posts_df, "num_posts"), (comments_df, "num_comments")):
        if df.empty:
            continue
        df["day"] = pd.to_datetime(df["day"], errors="coerce", utc=True)
        df["day"] = df["day"].dt.tz_convert(None).dt.normalize()
        df["user_id"] = df["user_id"].astype("string").str.strip()
        df[col] = df[col].astype("int32")
        df.dropna(subset=["day", "user_id"], inplace=True)
        df.sort_values(["day", "user_id"], inplace=True)
        df.reset_index(drop=True, inplace=True)

    # If replies_df is provided, we add the replies to the comment count
    if replies_df is not None and not replies_df.empty:
        replies_copy = replies_df[["day", "author_id"]].copy()
        replies_copy["day"] = pd.to_datetime(replies_copy["day"], errors="coerce", utc=True)
        replies_copy["day"] = replies_copy["day"].dt.tz_convert(None).dt.normalize()
        replies_copy["author_id"] = replies_copy["author_id"].astype("string").str.strip()
        replies_copy = replies_copy.dropna(subset=["day", "author_id"])
        
        # We aggregate the replies by (day, author_id)
        replies_counts = (
            replies_copy.groupby(["day", "author_id"], as_index=False)
            .size()
            .rename(columns={"size": "num_replies"})
        )
        
        # We rename author_id to user_id
        replies_counts = replies_counts.rename(columns={"author_id": "user_id"})
        
        # We merge the replies with the comments dataframe
        if not comments_df.empty:
            # Merge to add the replies to the existing comments.
            comments_df = comments_df.merge(
                replies_counts,
                on=["day", "user_id"],
                how="outer",
            )
            # We sum comments and replies.
            comments_df["num_comments"] = (
                comments_df["num_comments"].fillna(0).astype("int32") +
                comments_df["num_replies"].fillna(0).astype("int32")
            )
            comments_df = comments_df.drop(columns=["num_replies"])
        else:
            # If there are no comments in the DB, we use only the replies
            replies_counts = replies_counts.rename(columns={"num_replies": "num_comments"})
            replies_counts["num_comments"] = replies_counts["num_comments"].astype("int32")
            comments_df = replies_counts
        
        # We normalize again
        if not comments_df.empty:
            comments_df["day"] = pd.to_datetime(comments_df["day"], errors="coerce", utc=True)
            comments_df["day"] = comments_df["day"].dt.tz_convert(None).dt.normalize()
            comments_df["user_id"] = comments_df["user_id"].astype("string").str.strip()
            comments_df["num_comments"] = comments_df["num_comments"].astype("int32")
            comments_df.dropna(subset=["day", "user_id"], inplace=True)
            comments_df.sort_values(["day", "user_id"], inplace=True)
            comments_df.reset_index(drop=True, inplace=True)

    return posts_df, comments_df


def _day_index(series: pd.Series, all_days: pd.DatetimeIndex) -> np.ndarray:
    vals = pd.to_datetime(series).to_numpy(dtype="datetime64[ns]")
    return np.searchsorted(all_days.to_numpy(dtype="datetime64[ns]"), vals).astype(np.int32)


def _fit_powerlaw_discrete_xmin(
    values: np.ndarray,
    min_tail: int = 50,
    alpha_bounds: tuple[float, float] = (1.01, 8.0),
) -> tuple[float, float]:
    """Discrete power-law fit with xmin selection via KS minimization."""
    x = np.asarray(values)
    if x.size == 0:
        return np.nan, np.nan

    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan

    x = x.astype(np.int64)
    x = x[x > 0]
    if x.size < min_tail:
        return np.nan, np.nan

    candidates = np.unique(x)
    max_valid_xmin = np.sort(x)[x.size - min_tail]
    candidates = candidates[candidates <= max_valid_xmin]

    best_ks = np.inf
    best_alpha = np.nan
    best_xmin = np.nan

    for xmin in candidates:
        tail = x[x >= xmin]
        n = tail.size
        if n < min_tail:
            continue

        sum_log = np.log(tail).sum()

        def nll(alpha: float) -> float:
            z = zeta(alpha, float(xmin))
            if not np.isfinite(z) or z <= 0:
                return np.inf
            return n * np.log(z) + alpha * sum_log

        res = minimize_scalar(nll, bounds=alpha_bounds, method="bounded")
        if not res.success:
            continue

        alpha = float(res.x)
        z0 = zeta(alpha, float(xmin))
        if not np.isfinite(z0) or z0 <= 0:
            continue

        uniq, counts = np.unique(tail, return_counts=True)
        emp_cdf = np.cumsum(counts) / n
        model_cdf = 1.0 - (zeta(alpha, uniq.astype(float) + 1.0) / z0)
        ks = float(np.max(np.abs(emp_cdf - model_cdf)))

        if ks < best_ks:
            best_ks = ks
            best_alpha = alpha
            best_xmin = float(xmin)

    return best_alpha, best_xmin


def _component_shape_metrics(sizes: list[int]) -> dict:
    if not sizes:
        return {"n_components": 0, "top1": 0, "top5_sum": 0, "rank_size_slope": np.nan}

    s = np.array(sorted(sizes, reverse=True), dtype=float)
    r = np.arange(1, len(s) + 1, dtype=float)

    if len(s) >= 3:
        lr = np.log10(r)
        ls = np.log10(s)
        slope = float(np.polyfit(lr, ls, 1)[0])
    else:
        slope = np.nan

    return {
        "n_components": int(len(s)),
        "top1": int(s[0]),
        "top5_sum": int(s[:5].sum()),
        "rank_size_slope": slope,
    }


def _array_stats(a: np.ndarray, percentiles: tuple[int, ...] = PERCENTILES) -> dict:
    if a.size == 0:
        out = {"mean": np.nan, "median": np.nan, "min": np.nan, "max": np.nan, "std": np.nan}
        for p in percentiles:
            out[f"p{p}"] = np.nan
        return out

    q = np.percentile(a, percentiles)
    out = {
        "mean": float(np.mean(a)),
        "median": float(np.median(a)),
        "min": float(np.min(a)),
        "max": float(np.max(a)),
        "std": float(np.std(a)),
    }
    for p, v in zip(percentiles, q):
        out[f"p{p}"] = float(v)
    return out


def _largest_cc_fraction_undirected(G_u: nx.Graph) -> float:
    n = G_u.number_of_nodes()
    if n == 0:
        return 0.0
    giant = max((len(c) for c in nx.connected_components(G_u)), default=0)
    return giant / n


def _robustness_snapshot(
    G_u: nx.Graph,
    G_d: nx.DiGraph,
    removal_percents: tuple[float, ...] = REMOVAL_PERCENTS,
    random_runs: int = ROBUSTNESS_RANDOM_RUNS,
    seed: int = 42,
) -> pd.DataFrame:
    n0 = G_u.number_of_nodes()
    if n0 == 0:
        return pd.DataFrame(columns=["attack", "percent_removed", "removed_nodes", "gc_fraction", "gc_fraction_mean", "gc_fraction_std"])

    rng = np.random.default_rng(seed)
    nodes = np.array(list(G_u.nodes()))

    indegree_ranked = [n for n, _ in sorted(G_d.in_degree(), key=lambda x: x[1], reverse=True)]
    outdegree_ranked = [n for n, _ in sorted(G_d.out_degree(), key=lambda x: x[1], reverse=True)]

    rows = []
    for pct in removal_percents:
        k = max(1, int(round(pct * n0)))

        gc_vals = []
        for _ in range(random_runs):
            removed = set(rng.choice(nodes, size=k, replace=False))
            Hr = G_u.copy()
            Hr.remove_nodes_from(removed)
            gc_vals.append(_largest_cc_fraction_undirected(Hr))

        gc_random_mean = float(np.mean(gc_vals))
        rows.append({
            "attack": "random",
            "percent_removed": pct,
            "removed_nodes": k,
            "gc_fraction": gc_random_mean,
            "gc_fraction_mean": gc_random_mean,
            "gc_fraction_std": float(np.std(gc_vals)),
        })

        removed_in = set(indegree_ranked[:k])
        H_in = G_u.copy()
        H_in.remove_nodes_from(removed_in)
        gc_in = _largest_cc_fraction_undirected(H_in)
        rows.append({
            "attack": "targeted_indegree",
            "percent_removed": pct,
            "removed_nodes": k,
            "gc_fraction": gc_in,
            "gc_fraction_mean": gc_in,
            "gc_fraction_std": 0.0,
        })

        removed_out = set(outdegree_ranked[:k])
        H_out = G_u.copy()
        H_out.remove_nodes_from(removed_out)
        gc_out = _largest_cc_fraction_undirected(H_out)
        rows.append({
            "attack": "targeted_outdegree",
            "percent_removed": pct,
            "removed_nodes": k,
            "gc_fraction": gc_out,
            "gc_fraction_mean": gc_out,
            "gc_fraction_std": 0.0,
        })

    return pd.DataFrame(rows)


In [ ]:
def run_historical_graph_pipeline_optimized(
    db_path: Path = DB_PATH,
    start_date: str = START_DATE,
    cutoff_date: str = CUTOFF_DATE,
    _excel_output: Path = EXCEL_OUTPUT,
) -> dict[str, pd.DataFrame]:
    """Historical analysis pipeline"""
    _excel_output.parent.mkdir(parents=True, exist_ok=True)
    with connect_sqlite_read_only(db_path) as conn:
        replies = fetch_recursive_replies(conn, start_date=start_date, cutoff_date=cutoff_date)
        interactions = fetch_interactions(conn, start_date=start_date, cutoff_date=cutoff_date, replies_df=replies)
        users = fetch_active_users(conn, start_date=start_date, cutoff_date=cutoff_date, replies_df=replies)
        
        posts_daily, comments_daily = fetch_daily_activity_counts(conn, start_date=start_date, cutoff_date=cutoff_date, replies_df=replies)
        author_metrics_db = fetch_daily_author_metrics(conn, start_date=start_date, cutoff_date=cutoff_date)

    if users.empty:
        raise ValueError("Empty users dataset: impossible to build the time series")

    # Build metrics (karma / follower means per day)
    metrics_parts = []
    if not author_metrics_db.empty:
        author_metrics_db = author_metrics_db.copy()
        author_metrics_db['day'] = pd.to_datetime(author_metrics_db['day'], errors='coerce', utc=True)
        author_metrics_db['day'] = author_metrics_db['day'].dt.tz_convert(None).dt.normalize()
        author_metrics_db['author_id'] = author_metrics_db['author_id'].astype('string').str.strip()
        metrics_parts.append(author_metrics_db[['day', 'author_id', 'karma', 'followerCount']])
    if not replies.empty:
        r = replies[['day', 'author_id', 'author_karma', 'author_followers']].copy()
        r['day'] = pd.to_datetime(r['day'], errors='coerce', utc=True)
        r['day'] = r['day'].dt.tz_convert(None).dt.normalize()
        r = r.rename(columns={'author_karma': 'karma', 'author_followers': 'followerCount'})
        r['author_id'] = r['author_id'].astype('string').str.strip()
        r['karma'] = pd.to_numeric(r['karma'], errors='coerce')
        r['followerCount'] = pd.to_numeric(r['followerCount'], errors='coerce')
        metrics_parts.append(r[['day', 'author_id', 'karma', 'followerCount']])

    if metrics_parts:
        combined_metrics = pd.concat(metrics_parts, ignore_index=True, sort=False)
        combined_metrics = combined_metrics.dropna(subset=['day', 'author_id'])
        combined_metrics['day'] = pd.to_datetime(combined_metrics['day'], errors='coerce', utc=True)
        combined_metrics['day'] = combined_metrics['day'].dt.tz_convert(None).dt.normalize()
        combined_metrics['karma'] = pd.to_numeric(combined_metrics['karma'], errors='coerce')
        combined_metrics['followerCount'] = pd.to_numeric(combined_metrics['followerCount'], errors='coerce')
        per_user = (
            combined_metrics.groupby(['day', 'author_id'], as_index=False)
            .agg({'karma': 'mean', 'followerCount': 'mean'})
        )
        day_means = per_user.groupby('day', as_index=True).agg({'karma': 'mean', 'followerCount': 'mean'})
    else:
        day_means = pd.DataFrame(columns=['karma', 'followerCount'])

    # Setup index structures
    # Include also authors seen in posts_daily and comments_daily and normalize ids
    parts = [
        users["user_id"].astype("string").str.strip(),
        interactions["source"].astype("string").str.strip(),
        interactions["target"].astype("string").str.strip(),
    ]
    if 'posts_daily' in locals() and not posts_daily.empty:
        parts.append(posts_daily["user_id"].astype("string").str.strip())
    if 'comments_daily' in locals() and not comments_daily.empty:
        parts.append(comments_daily["user_id"].astype("string").str.strip())

    all_user_ids = pd.Index(pd.unique(pd.concat(parts, ignore_index=True).dropna()))
    user_to_idx = pd.Series(np.arange(len(all_user_ids), dtype=np.int32), index=all_user_ids)

    start_day = min(
        users["day"].min(),
        interactions["day"].min() if not interactions.empty else users["day"].min(),
    ) if 'start_date' not in locals() else pd.to_datetime(start_date)
    end_day = pd.to_datetime(cutoff_date)
    all_days = pd.date_range(start_day, end_day, freq="D")
    day_count = len(all_days)

    # Prepare data
    users2 = users.copy()
    users2["didx"] = _day_index(users2["day"], all_days)
    users2["uidx"] = user_to_idx.loc[users2["user_id"]].to_numpy(dtype=np.int32)
    users2 = users2.sort_values(["didx", "uidx"]).reset_index(drop=True)

    inter2 = interactions.copy()
    if not inter2.empty:
        inter2["didx"] = _day_index(inter2["day"], all_days)
        inter2["sidx"] = user_to_idx.loc[inter2["source"]].to_numpy(dtype=np.int32)
        inter2["tidx"] = user_to_idx.loc[inter2["target"]].to_numpy(dtype=np.int32)
        inter2 = inter2.sort_values(["didx", "sidx", "tidx"]).reset_index(drop=True)

    def prep_activity(df: pd.DataFrame, col: str) -> pd.DataFrame:
        if df.empty:
            return pd.DataFrame(columns=["didx", "uidx", col])
        x = df.copy()
        x["didx"] = _day_index(x["day"], all_days)
        x["uidx"] = user_to_idx.loc[x["user_id"]].to_numpy(dtype=np.int32)
        x = x[["didx", "uidx", col]].sort_values(["didx", "uidx"]).reset_index(drop=True)
        return x

    posts2 = prep_activity(posts_daily, "num_posts")
    comm2 = prep_activity(comments_daily, "num_comments")

    # Main loop
    n_users = len(all_user_ids)
    active_mask = np.zeros(n_users, dtype=bool)
    active_indices: list[int] = []
    post_counts = np.zeros(n_users, dtype=np.int32)
    comment_counts = np.zeros(n_users, dtype=np.int32)
    indeg = np.zeros(n_users, dtype=np.int32)
    outdeg = np.zeros(n_users, dtype=np.int32)
    instrength = np.zeros(n_users, dtype=np.int64)

    seen_edges: set[tuple[int, int]] = set()
    edge_weights: dict[tuple[int, int], int] = {}
    total_weight_interactions = 0
    non_isolated = 0

    G_d = nx.DiGraph()
    G_u = nx.Graph()
    G_dw = nx.DiGraph()

    daily_rows = []
    robustness_rows = []

    prev_alpha_indeg = np.nan
    prev_xmin_indeg = np.nan
    prev_alpha_instr = np.nan
    prev_xmin_instr = np.nan
    prev_giant_wcc_pct = np.nan
    prev_giant_scc_pct = np.nan
    prev_wshape = _component_shape_metrics([])
    prev_sshape = _component_shape_metrics([])
    prev_robustness_template = pd.DataFrame()

    pu = pi = pp = pc = 0
    users_arr = users2[["didx", "uidx"]].to_numpy(dtype=np.int32)
    inter_arr = inter2[["didx", "sidx", "tidx"]].to_numpy(dtype=np.int32) if not inter2.empty else np.empty((0, 3), dtype=np.int32)
    posts_arr = posts2[["didx", "uidx", "num_posts"]].to_numpy(dtype=np.int32) if not posts2.empty else np.empty((0, 3), dtype=np.int32)
    comm_arr = comm2[["didx", "uidx", "num_comments"]].to_numpy(dtype=np.int32) if not comm2.empty else np.empty((0, 3), dtype=np.int32)

    for d in range(day_count):
        day = all_days[d]
        nodes_added_today = 0

        while pu < len(users_arr) and users_arr[pu, 0] == d:
            u = int(users_arr[pu, 1])
            if not active_mask[u]:
                active_mask[u] = True
                active_indices.append(u)
                nodes_added_today += 1
                G_d.add_node(u)
                G_u.add_node(u)
                G_dw.add_node(u)
            pu += 1

        while pp < len(posts_arr) and posts_arr[pp, 0] == d:
            u = int(posts_arr[pp, 1])
            inc = int(posts_arr[pp, 2])
            post_counts[u] += inc
            pp += 1

        while pc < len(comm_arr) and comm_arr[pc, 0] == d:
            u = int(comm_arr[pc, 1])
            inc = int(comm_arr[pc, 2])
            comment_counts[u] += inc
            pc += 1

        edges_added_today = 0
        weight_added_today = 0
        while pi < len(inter_arr) and inter_arr[pi, 0] == d:
            s = int(inter_arr[pi, 1])
            t = int(inter_arr[pi, 2])
            e = (s, t)

            w_new = edge_weights.get(e, 0) + 1
            edge_weights[e] = w_new
            instrength[t] += 1
            weight_added_today += 1
            total_weight_interactions += 1

            if w_new == 1:
                edges_added_today += 1

                if (indeg[s] + outdeg[s]) == 0 and active_mask[s]:
                    non_isolated += 1
                if (indeg[t] + outdeg[t]) == 0 and active_mask[t] and t != s:
                    non_isolated += 1

                outdeg[s] += 1
                indeg[t] += 1
                seen_edges.add(e)

                G_d.add_edge(s, t)
                G_u.add_edge(s, t)
                G_dw.add_edge(s, t, weight=1)
            else:
                G_dw[s][t]["weight"] = w_new

            pi += 1

        graph_changed_today = (nodes_added_today > 0) or (edges_added_today > 0)

        if active_indices:
            active_idx_today = np.fromiter(active_indices, dtype=np.int32)
            n_nodes = int(active_idx_today.size)
        else:
            active_idx_today = np.array([], dtype=np.int32)
            n_nodes = 0

        n_edges_directed_unique = len(seen_edges)
        n_edges_undirected_unique = G_u.number_of_edges()
        edge_weight_total = int(total_weight_interactions)
        n_isolates = int(max(n_nodes - non_isolated, 0))

        links_per_node = (n_edges_undirected_unique / n_nodes) if n_nodes > 0 else np.nan
        avg_degree = (2.0 * n_edges_undirected_unique / n_nodes) if n_nodes > 0 else np.nan

        if n_nodes > 0:
            posts_active = post_counts[active_idx_today]
            comm_active = comment_counts[active_idx_today]
            posts_total = int(posts_active.sum())
            comments_total = int(comm_active.sum())
            pst = _array_stats(posts_active)
            cst = _array_stats(comm_active)

            indeg_active = indeg[active_idx_today]
            instr_active = instrength[active_idx_today]
            if edges_added_today > 0:
                alpha_indeg, xmin_indeg = _fit_powerlaw_discrete_xmin(indeg_active)
                alpha_instr, xmin_instr = _fit_powerlaw_discrete_xmin(instr_active)
                prev_alpha_indeg, prev_xmin_indeg = alpha_indeg, xmin_indeg
                prev_alpha_instr, prev_xmin_instr = alpha_instr, xmin_instr
            else:
                alpha_indeg, xmin_indeg = prev_alpha_indeg, prev_xmin_indeg
                alpha_instr, xmin_instr = prev_alpha_instr, prev_xmin_instr

            if graph_changed_today:
                wcc_sizes = sorted((len(c) for c in nx.connected_components(G_u)), reverse=True)
                scc_sizes = sorted((len(c) for c in nx.strongly_connected_components(G_d)), reverse=True)

                giant_wcc_pct = (wcc_sizes[0] / n_nodes * 100.0) if wcc_sizes else 0.0
                giant_scc_pct = (scc_sizes[0] / n_nodes * 100.0) if scc_sizes else 0.0

                wshape = _component_shape_metrics(wcc_sizes)
                sshape = _component_shape_metrics(scc_sizes)

                prev_giant_wcc_pct = giant_wcc_pct
                prev_giant_scc_pct = giant_scc_pct
                prev_wshape = wshape
                prev_sshape = sshape

                prev_robustness_template = _robustness_snapshot(G_u, G_d, REMOVAL_PERCENTS, ROBUSTNESS_RANDOM_RUNS)
            else:
                giant_wcc_pct = prev_giant_wcc_pct
                giant_scc_pct = prev_giant_scc_pct
                wshape = prev_wshape
                sshape = prev_sshape

            if not prev_robustness_template.empty:
                rob_day = prev_robustness_template.copy()
                rob_day.insert(0, "day", day)
                robustness_rows.append(rob_day)
        else:
            posts_total = 0
            comments_total = 0
            pst = _array_stats(np.array([], dtype=float))
            cst = _array_stats(np.array([], dtype=float))
            alpha_indeg = np.nan
            xmin_indeg = np.nan
            alpha_instr = np.nan
            xmin_instr = np.nan
            giant_wcc_pct = np.nan
            giant_scc_pct = np.nan
            wshape = _component_shape_metrics([])
            sshape = _component_shape_metrics([])

        # Karma / follower summary (daily + cumulative)
        if day in day_means.index:
            avg_karma_nodes = float(day_means.loc[day, 'karma']) if pd.notna(day_means.loc[day, 'karma']) else np.nan
            avg_follower_nodes = float(day_means.loc[day, 'followerCount']) if pd.notna(day_means.loc[day, 'followerCount']) else np.nan
        else:
            avg_karma_nodes = np.nan
            avg_follower_nodes = np.nan

        try:
            avg_karma_nodes_cum = float(day_means.loc[:day, 'karma'].mean()) if not day_means.empty else np.nan
        except Exception:
            avg_karma_nodes_cum = np.nan
        avg_karma_nodes_cum_full = float(day_means_cum_full.loc[day, 'karma']) if 'day_means_cum_full' in globals() and day in day_means_cum_full.index else np.nan

        try:
            avg_follower_nodes_cum = float(day_means.loc[:day, 'followerCount'].mean()) if not day_means.empty else np.nan
        except Exception:
            avg_follower_nodes_cum = np.nan
        avg_follower_nodes_cum_full = float(day_means_cum_full.loc[day, 'followerCount']) if 'day_means_cum_full' in globals() and day in day_means_cum_full.index else np.nan

        daily_rows.append({
            "day": day,
            "nodes": n_nodes,
            "isolated_nodes": n_isolates,
            "edges_unique": n_edges_directed_unique,
            "edges_unique_directed": n_edges_directed_unique,
            "edges_unique_undirected": n_edges_undirected_unique,
            "edge_weight_total": edge_weight_total,
            "avg_degree": avg_degree,
            "links_per_node": links_per_node,
            "edges_added_today_unique": edges_added_today,
            "edge_weight_added_today": weight_added_today,
            "alpha_indegree": alpha_indeg,
            "xmin_indegree": xmin_indeg,
            "alpha_instrength": alpha_instr,
            "xmin_instrength": xmin_instr,
            "giant_wcc_pct": giant_wcc_pct,
            "giant_scc_pct": giant_scc_pct,
            "wcc_n_components": wshape["n_components"],
            "wcc_top1": wshape["top1"],
            "wcc_top5_sum": wshape["top5_sum"],
            "wcc_rank_size_slope": wshape["rank_size_slope"],
            "scc_n_components": sshape["n_components"],
            "scc_top1": sshape["top1"],
            "scc_top5_sum": sshape["top5_sum"],
            "scc_rank_size_slope": sshape["rank_size_slope"],
            "posts_total": posts_total,
            "posts_mean": pst["mean"],
            "posts_median": pst["median"],
            "posts_min": pst["min"],
            "posts_max": pst["max"],
            "posts_std": pst["std"],
            "posts_p5": pst["p5"],
            "posts_p25": pst["p25"],
            "posts_p50": pst["p50"],
            "posts_p75": pst["p75"],
            "posts_p95": pst["p95"],
            "comments_total": comments_total,
            "comments_mean": cst["mean"],
            "comments_median": cst["median"],
            "comments_min": cst["min"],
            "comments_max": cst["max"],
            "comments_std": cst["std"],
            "comments_p5": cst["p5"],
            "comments_p25": cst["p25"],
            "comments_p50": cst["p50"],
            "comments_p75": cst["p75"],
            "comments_p95": cst["p95"],
            "avg_karma_nodes": avg_karma_nodes,
            "avg_karma_nodes_cum": avg_karma_nodes_cum,
            "avg_karma_nodes_cum_full": avg_karma_nodes_cum_full,
            "avg_follower_nodes": avg_follower_nodes,
            "avg_follower_nodes_cum": avg_follower_nodes_cum,
            "avg_follower_nodes_cum_full": avg_follower_nodes_cum_full,
        })

    results = {
        'daily_all_metrics': pd.DataFrame(daily_rows),
        'robustness_daily': pd.concat(robustness_rows, ignore_index=True) if robustness_rows else pd.DataFrame(),
    }

    return results

In [ ]:
# === OPTIMIZED VERSIONS (override old functions) ===

def fetch_daily_activity_counts_opt(
    conn: sqlite3.Connection,
    start_date: str = START_DATE,
    cutoff_date: str = CUTOFF_DATE,
    replies_df: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """OPTIMIZED: Usa UNION in SQL per ridurre n° query e pandas overhead.
    Include anche i commenti annidati (replies) se disponibili."""
    query = f"""
        SELECT DATE(p.created_at) AS day,
               CAST(p.author_id AS TEXT) AS user_id,
               COUNT(*) AS num_posts,
               0 AS num_comments
        FROM posts AS p
        WHERE DATE(p.created_at) >= ?
          AND DATE(p.created_at) <= ?
          AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL
        GROUP BY DATE(p.created_at), CAST(p.author_id AS TEXT)
        
        UNION ALL
        
        SELECT DATE(c.created_at) AS day,
               CAST(c.author_id AS TEXT) AS user_id,
               0 AS num_posts,
               COUNT(*) AS num_comments
        FROM comments_new AS c
        WHERE DATE(c.created_at) >= ?
          AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')}
          AND c.author_id IS NOT NULL
        GROUP BY DATE(c.created_at), CAST(c.author_id AS TEXT)
    """
    
    combined_df = pd.read_sql_query(query, conn, params=(start_date, cutoff_date, start_date, cutoff_date))
    
    # Aggiungi commenti annidati (reply) se disponibili
    if replies_df is not None and not replies_df.empty:
        reply_activity = replies_df[['day', 'author_id']].copy()
        reply_activity = reply_activity.rename(columns={'author_id': 'user_id'})
        reply_activity['num_posts'] = 0
        reply_activity['num_comments'] = 1
        combined_df = pd.concat([combined_df, reply_activity], ignore_index=True, sort=False)
    
    if combined_df.empty:
        return pd.DataFrame(columns=["day", "user_id", "num_posts"]), pd.DataFrame(columns=["day", "user_id", "num_comments"])
    
    combined_df["day"] = pd.to_datetime(combined_df["day"], errors="coerce", utc=True)
    combined_df["day"] = combined_df["day"].dt.tz_convert(None).dt.normalize()
    combined_df["user_id"] = combined_df["user_id"].astype("string").str.strip()
    
    aggregated = combined_df.groupby(["day", "user_id"], as_index=False).agg({
        "num_posts": "sum",
        "num_comments": "sum"
    }).astype({"num_posts": "int32", "num_comments": "int32"})
    
    aggregated.dropna(subset=["day", "user_id"], inplace=True)
    aggregated.sort_values(["day", "user_id"], inplace=True)
    aggregated.reset_index(drop=True, inplace=True)
    
    posts_df = aggregated[aggregated["num_posts"] > 0][["day", "user_id", "num_posts"]].copy()
    comments_df = aggregated[aggregated["num_comments"] > 0][["day", "user_id", "num_comments"]].copy()

    return posts_df, comments_df


fetch_daily_activity_counts = fetch_daily_activity_counts_opt

In [ ]:
# === OPTIMIZATION: Parquet-first reading ===
# Monkey-patch fetch functions to use parquet if available

_fetch_interactions_original = fetch_interactions
_fetch_active_users_original = fetch_active_users

def fetch_interactions_optimized(conn, start_date, cutoff_date, replies_df=None):
    """Use Parquet if available, otherwise query the database."""
    parquet_path = OUTPUT_DIR / 'interactions.parquet'
    if parquet_path.exists():
        print(f"  💾 interactions.parquet found, reading from file...")
        df = pd.read_parquet(parquet_path)
        df['day'] = pd.to_datetime(df['day'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
        df['source'] = df['source'].astype('string').str.strip()
        df['target'] = df['target'].astype('string').str.strip()
        return df[(df['day'] >= start_date) & (df['day'] <= cutoff_date)].reset_index(drop=True)
    print(f"  🔍 Fetch interactions da DB...")
    return _fetch_interactions_original(conn, start_date, cutoff_date, replies_df)

def fetch_active_users_optimized(conn, start_date, cutoff_date, replies_df=None):
    """Use Parquet if available, otherwise query the database."""
    parquet_path = OUTPUT_DIR / 'users.parquet'
    if parquet_path.exists():
        print(f"  💾 users.parquet found, reading from file...")
        df = pd.read_parquet(parquet_path)
        df['day'] = pd.to_datetime(df['day'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
        df['user_id'] = df['user_id'].astype('string').str.strip()
        return df[(df['day'] >= start_date) & (df['day'] <= cutoff_date)].reset_index(drop=True)
    print(f"  🔍 Fetch users da DB...")
    return _fetch_active_users_original(conn, start_date, cutoff_date, replies_df)

# Enable optimization.
fetch_interactions = fetch_interactions_optimized
fetch_active_users = fetch_active_users_optimized
print("✨ Parquet optimization enabled: fetch_interactions, fetch_active_users")

In [ ]:
def build_day_means_cum_full(
    db_path: Path,
    start_date: str,
    cutoff_date: str,
) -> pd.DataFrame:
    """Builds the full cumulative average of karma and followerCount by day."""
    with connect_sqlite_read_only(db_path) as conn:
        replies = fetch_recursive_replies(conn, start_date=start_date, cutoff_date=cutoff_date)
        author_metrics_db = fetch_daily_author_metrics(conn, start_date=start_date, cutoff_date=cutoff_date)

    metrics_parts = []
    if not author_metrics_db.empty:
        author_metrics_db = author_metrics_db.copy()
        author_metrics_db['day'] = pd.to_datetime(author_metrics_db['day'], errors='coerce', utc=True)
        author_metrics_db['day'] = author_metrics_db['day'].dt.tz_convert(None).dt.normalize()
        author_metrics_db['author_id'] = author_metrics_db['author_id'].astype('string').str.strip()
        metrics_parts.append(author_metrics_db[['day', 'author_id', 'karma', 'followerCount']])

    if not replies.empty:
        r = replies[['day', 'author_id', 'author_karma', 'author_followers']].copy()
        r['day'] = pd.to_datetime(r['day'], errors='coerce', utc=True)
        r['day'] = r['day'].dt.tz_convert(None).dt.normalize()
        r = r.rename(columns={'author_karma': 'karma', 'author_followers': 'followerCount'})
        r['author_id'] = r['author_id'].astype('string').str.strip()
        r['karma'] = pd.to_numeric(r['karma'], errors='coerce')
        r['followerCount'] = pd.to_numeric(r['followerCount'], errors='coerce')
        metrics_parts.append(r[['day', 'author_id', 'karma', 'followerCount']])

    if not metrics_parts:
        return pd.DataFrame(columns=['karma', 'followerCount'])

    combined_metrics = pd.concat(metrics_parts, ignore_index=True, sort=False)
    combined_metrics = combined_metrics.dropna(subset=['day', 'author_id'])
    combined_metrics['day'] = pd.to_datetime(combined_metrics['day'], errors='coerce', utc=True)
    combined_metrics['day'] = combined_metrics['day'].dt.tz_convert(None).dt.normalize()
    combined_metrics['karma'] = pd.to_numeric(combined_metrics['karma'], errors='coerce')
    combined_metrics['followerCount'] = pd.to_numeric(combined_metrics['followerCount'], errors='coerce')

    per_user = (
        combined_metrics.groupby(['day', 'author_id'], as_index=False)
        .agg({'karma': 'mean', 'followerCount': 'mean'})
    )
    if per_user.empty:
        return pd.DataFrame(columns=['karma', 'followerCount'])

    day_grid = pd.date_range(pd.to_datetime(start_date), pd.to_datetime(cutoff_date), freq='D')
    cumulative_parts = []

    for author_id, author_frame in per_user.groupby('author_id', sort=False):
        author_series = (
            author_frame.sort_values('day')
            .set_index('day')[['karma', 'followerCount']]
            .reindex(day_grid)
            .ffill()
        )
        first_seen_day = author_frame['day'].min()
        author_series = author_series.loc[first_seen_day:]
        author_series = author_series.reset_index().rename(columns={'index': 'day'})
        author_series['author_id'] = author_id
        cumulative_parts.append(author_series)

    per_user_cum_full = pd.concat(cumulative_parts, ignore_index=True, sort=False)
    day_means_cum_full = per_user_cum_full.groupby('day', as_index=True).agg({'karma': 'mean', 'followerCount': 'mean'})
    day_means_cum_full = day_means_cum_full.sort_index()
    return day_means_cum_full


day_means_cum_full = build_day_means_cum_full(DB_PATH, START_DATE, CUTOFF_DATE)

In [ ]:
from openpyxl import Workbook

# Ensure Excel file exists with at least one visible sheet to avoid openpyxl IndexError
EXCEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
if not EXCEL_OUTPUT.exists():
    wb = Workbook()
    ws = wb.active
    ws.title = "placeholder"
    wb.save(EXCEL_OUTPUT)

# 1) Builds and saves the complete dynamic metrics. (sheet: daily_all_metrics, robustness_daily)
results = run_historical_graph_pipeline_optimized(
    db_path=DB_PATH,
    start_date=START_DATE,
    cutoff_date=CUTOFF_DATE,
    _excel_output=EXCEL_OUTPUT,
)

# 2) Calculates and saves the preferential attachment metrics
pref_attach_timeseries, pref_attach_summary = compute_preferential_attachment_timeseries(OUTPUT_DIR)

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # write/replace the dynamic metric sheets first
    if isinstance(results, dict):
        results['daily_all_metrics'].to_excel(writer, sheet_name='daily_all_metrics', index=False)
        results['robustness_daily'].to_excel(writer, sheet_name='robustness_daily', index=False)

    # write/replace preferential-attachment sheets
    pref_attach_timeseries.to_excel(writer, sheet_name='pref_attach_timeseries', index=False)
    pref_attach_summary.to_excel(writer, sheet_name='pref_attach_summary', index=False)

# Display brief previews
if isinstance(results, dict):
    display(results.get('daily_all_metrics').head() if not results.get('daily_all_metrics').empty else results.get('daily_all_metrics'))
    display(results.get('robustness_daily').head() if not results.get('robustness_daily').empty else results.get('robustness_daily'))

display(pref_attach_timeseries.head())
display(pref_attach_summary.head())


In [ ]:
import pandas as pd

users_source = users_df if 'users_df' in globals() and not users_df.empty else pd.read_parquet(OUTPUT_DIR / 'users.parquet', columns=['day', 'user_id'])

users_source = users_source[['day', 'user_id']].copy()
users_source['day'] = pd.to_datetime(users_source['day'], errors='coerce', utc=True)
users_source['day'] = users_source['day'].dt.tz_convert(None).dt.normalize()
users_source['user_id'] = users_source['user_id'].astype('string').str.strip()

active_nodes_today = (
    users_source.dropna(subset=['day', 'user_id'])
    .groupby('day', as_index=False)['user_id']
    .nunique()
    .rename(columns={'user_id': 'Active_nodes_today'})
    .sort_values('day')
    .reset_index(drop=True)
)

display(active_nodes_today)